O objetivo é encontrar uma definicação ideal apra o formato dos superpixel, que vão virar os nós da rede

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage.segmentation import slic, mark_boundaries
import glob
import numpy as np
import pandas as pd
from PIL import Image
from skimage.color import rgb2gray
from skimage.feature import canny
from skimage.segmentation import slic, find_boundaries
from scipy.ndimage import distance_transform_edt

In [ ]:
# Função para carregar a imagem e redimensionar para o padrão combinado com o Luiz
def carregar_imagem(caminho):
    img = Image.open(caminho).convert("RGB").resize((224, 224))
    return np.array(img)


#  Exemplo com a benigna

caminho_img = "../data/original_data/benign_lesions/1.jpg" 
img_np = carregar_imagem(caminho_img)

# Testando diferentes granularidades e formatos
n_segments_list = [50, 80, 120]
compactness_list = [10, 15, 20]

fig, axes = plt.subplots(len(n_segments_list), len(compactness_list), figsize=(12, 12))

for i, n_seg in enumerate(n_segments_list):
    for j, comp in enumerate(compactness_list):
        # start_label=0 é vital para mapearmos os IDs dos nós de 0 a N amanhã
        segments = slic(img_np, n_segments=n_seg, compactness=comp, start_label=0)
        
        ax = axes[i, j]
        ax.imshow(mark_boundaries(img_np, segments))
        ax.set_title(f"Segs: {n_seg} | Comp: {comp}")
        ax.axis("off")

plt.tight_layout()
plt.show()

O melhor visaulmente foi o Segs 80 e Comp 15

In [ ]:
# Gerando a segmentação final com os parâmetros escolhidos
segments_final = slic(img_np, n_segments=80, compactness=15, start_label=0)

# Mapeamento região -> pixels (coração da nossa perturbação)
num_nodes = segments_final.max() + 1
regiao_para_pixels = {node_id: np.where(segments_final == node_id) for node_id in range(num_nodes)}

# Copia a imagem original para mascarar
img_mascarada = img_np.copy()

# Escolhendo uma região arbitrária para desligar (ex: id 15)
id_regiao = 32
img_mascarada[regiao_para_pixels[id_regiao]] = [0, 0, 0] # Pinta de preto

# Visualiza o resultado
plt.figure(figsize=(4, 4))
plt.imshow(img_mascarada)
plt.title(f"Região {id_regiao} Mascarada")
plt.axis("off")
plt.show()

In [ ]:
def boundary_recall(img_np: np.ndarray, segments: np.ndarray, tol: int = 2) -> float:
    """Calcula a fração de bordas (Canny) cobertas pelas fronteiras SLIC dentro de `tol` pixels."""
    gray = rgb2gray(img_np)
    ref_edges = canny(gray, sigma=1.5)
    if not np.any(ref_edges):
        return 0.0
    
    slic_bounds = find_boundaries(segments, mode="thick")
    # Distância de cada pixel até a fronteira SLIC mais próxima
    dist_to_slic = distance_transform_edt(~slic_bounds)
    
    matched_edges = ref_edges & (dist_to_slic <= tol)
    return float(matched_edges.sum() / ref_edges.sum())

# Amostra de 10 imagens (5 benignas, 5 malignas) para calibrar os hiperparâmetros
amostras = (
    sorted(glob.glob("../data/original_data/benign_lesions/*.jpg"))[:5] +
    sorted(glob.glob("../data/original_data/malignant_lesions/*.jpg"))[:5]
)

resultados = []
for n_seg in [40, 70, 100]:
    for comp in [5, 10, 20]:
        recalls, nos_reais = [], []
        for path in amostras:
            img = np.asarray(Image.open(path).convert("RGB").resize((224, 224)), dtype=np.float32) / 255.0
            segs = slic(img, n_segments=n_seg, compactness=comp, start_label=0)
            recalls.append(boundary_recall(img, segs, tol=2))
            nos_reais.append(int(segs.max() + 1))
            
        resultados.append({
            "n_segments": n_seg,
            "compactness": comp,
            "nos_medios": round(np.mean(nos_reais), 1),
            "boundary_recall_mean": round(np.mean(recalls), 4),
            "boundary_recall_std": round(np.std(recalls), 4),
        })

df_slic = pd.DataFrame(resultados)
display(df_slic)

 (70, 10): Tem uma média de 57,5 nós e um boundary recall de 0.6258 ($\pm 0.0511$). É um grafo bem enxuto e rápido de processar na API, mas deixa escapar cerca de 37% das bordas de referência

 (100, 20): Tem uma média de 98,3 nós e um boundary recall de 0.6237 ($\pm 0.0439$).

In [ ]:
N_SEGMENTS, COMPACTNESS= 70, 10 

In [ ]:
import sys
import importlib
sys.path.append("..")  # garante que o Python encontre a pasta src/

import src.explainability
importlib.reload(src.explainability)

import torch
import matplotlib.pyplot as plt
from PIL import Image
from src.modelo_treinado import CNN
from src.explainability import SuperpixelGraphExplainer
import importlib
import src.modelo_treinado
importlib.reload(src.modelo_treinado)
from src.modelo_treinado import CNNLightning


model = CNNLightning.load_from_checkpoint("../models/cnn_geral.ckpt", map_location="cpu")
explainer = SuperpixelGraphExplainer(model, n_segments=70, compactness=10, num_samples=150)

amostras_visuais = [
    
    ("Benigno 1", "../data/augmented_data/augmented_benign/1.jpg"),
    ("Benigno 2", "../data/augmented_data/augmented_benign/2.jpg"),
    ("Maligno 1", "../data/augmented_data/augmented_malignant/1.jpg"),
    ("Maligno 2", "../data/augmented_data/augmented_malignant/2.jpg"),
]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, f1_score

def plotar_e_definir_cortes(y_true, y_prob, recall_alvo=0.98):
    # precision_recall_curve retorna thresholds em ordem crescente;
    # precision e recall têm 1 elemento a mais que thresholds
    prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
    prec_t = prec[:-1]
    rec_t = rec[:-1]

    # Calcula F1 para cada limiar avaliado
    f1_t = np.where(
        (prec_t + rec_t) > 0,
        2 * (prec_t * rec_t) / (prec_t + rec_t),
        0.0
    )

    # 1. t_baixo: maior limiar que ainda mantém recall >= recall_alvo (evita FN)
    idx_rec = np.where(rec_t >= recall_alvo)[0]
    if len(idx_rec) > 0:
        # Escolhe o maior limiar dentro da meta de recall para não jogar benignos demais em dúvida
        best_i_baixo = idx_rec[np.argmax(thresholds[idx_rec])]
        t_baixo = float(thresholds[best_i_baixo])
    else:
        t_baixo = float(thresholds[np.argmax(rec_t)])

    # 2. t_alto: limiar que maximiza o F1-score (garantindo que fique acima de t_baixo)
    mask_alto = thresholds > t_baixo
    if np.any(mask_alto):
        idx_candidates = np.where(mask_alto)[0]
        best_i_alto = idx_candidates[np.argmax(f1_t[idx_candidates])]
        t_alto = float(thresholds[best_i_alto])
    else:
        t_alto = min(float(t_baixo + 0.15), 0.90)

    # Plot das curvas Precision e Recall em função do limiar
    plt.figure(figsize=(9, 5))
    plt.plot(thresholds, prec_t, label="Precision (classe Maligno)", linewidth=2)
    plt.plot(thresholds, rec_t, label="Recall (classe Maligno)", linewidth=2)
    plt.plot(thresholds, f1_t, label="F1-score", linestyle="--", alpha=0.8)

    plt.axvline(t_baixo, color="green", linestyle=":", linewidth=2, label=f"t_baixo = {t_baixo:.3f}")
    plt.axvline(t_alto, color="red", linestyle=":", linewidth=2, label=f"t_alto = {t_alto:.3f}")

    plt.axvspan(0.0, t_baixo, color="green", alpha=0.06)
    plt.axvspan(t_baixo, t_alto, color="orange", alpha=0.08)
    plt.axvspan(t_alto, 1.0, color="red", alpha=0.06)

    plt.title("Precision, Recall e F1 em função do limiar (Validação)")
    plt.xlabel("Limiar de probabilidade")
    plt.ylabel("Pontuação")
    plt.xlim(0.0, 1.0)
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

    return t_baixo, t_alto

def classificar_tres_faixas(prob, t_baixo, t_alto):
    if prob >= t_alto:
        return "Maligno"
    elif prob >= t_baixo:
        return "Inconclusivo"
    else:
        return "Benigno"

# Exemplo de uso:
# t_baixo, t_alto = plotar_e_definir_cortes(y_true_val, y_prob_val, recall_alvo=0.98)
# print(f"Faixas: Benigno < {t_baixo:.3f} | Inconclusivo [{t_baixo:.3f}, {t_alto:.3f}) | Maligno >= {t_alto:.3f}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import recall_score, precision_score

# 1. Ajuste os caminhos das pastas do seu dataset (validação ou original_data)
PASTA_BENIGNO = Path("../data/original_data/benign_lesions")      # rótulo 0
PASTA_MALIGNO = Path("../data/original_data/malignant_lesions")   # rótulo 1

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

amostras = (
    [(p, 0) for p in PASTA_BENIGNO.rglob("*") if p.suffix.lower() in EXTS]
    + [(p, 1) for p in PASTA_MALIGNO.rglob("*") if p.suffix.lower() in EXTS]
)
print(f"Total de imagens carregadas: {len(amostras)} "
      f"(Benigno: {sum(y == 0 for _, y in amostras)} | Maligno: {sum(y == 1 for _, y in amostras)})")

# 2. Dataset e DataLoader com o mesmo tamanho de entrada do modelo/explainer
W, H = explainer.input_width, explainer.input_height  # ex: 304 x 304

transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    # Descomente a linha abaixo se o seu CNNLightning foi treinado com normalização ImageNet:
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class OralDataset(Dataset):
    def __init__(self, itens, tfm):
        self.itens = itens
        self.tfm = tfm

    def __len__(self):
        return len(self.itens)

    def __getitem__(self, idx):
        caminho, label = self.itens[idx]
        img = Image.open(caminho).convert("RGB")
        return self.tfm(img), label

loader = DataLoader(OralDataset(amostras, transform), batch_size=32, shuffle=False)

# 3. Inferência em lote para extrair y_true_val e y_prob_val
device = next(model.parameters()).device
model.eval()

y_true_list, y_prob_list = [], []

with torch.no_grad():
    for x, y in loader:
        x = x.to(device)
        out = model(x)

        # Trata saída com 1 logit (BCE) ou 2 logits (CrossEntropy)
        if out.ndim == 2 and out.shape[1] == 2:
            probs = torch.softmax(out, dim=1)[:, 1]
        else:
            # Se o forward() do seu modelo já aplicar sigmoid, troque por: probs = out.view(-1)
            probs = probs = out.view(-1)
        y_prob_list.append(probs.cpu().numpy())
        y_true_list.append(y.numpy())

y_true_val = np.concatenate(y_true_list)
y_prob_val = np.concatenate(y_prob_list)

# 4. Define os cortes pela curva PR (usando a função plotar_e_definir_cortes da célula anterior)
t_baixo, t_alto = plotar_e_definir_cortes(y_true_val, y_prob_val, recall_alvo=0.95)
print(f"\nLimiares sugeridos -> Benigno < {t_baixo:.3f} | Inconclusivo [{t_baixo:.3f}, {t_alto:.3f}) | Maligno >= {t_alto:.3f}")

# 5. Avalia a classificação em 3 faixas no dataset
faixas_val = np.array([classificar_tres_faixas(p, t_baixo, t_alto) for p in y_prob_val])

# Proteção contra falsos negativos: tudo que NÃO caiu em "Benigno" vai para revisão/alerta
nao_benigno_pred = (y_prob_val >= t_baixo).astype(int)
maligno_direto_pred = (y_prob_val >= t_alto).astype(int)

fn_benigno = int(((y_true_val == 1) & (faixas_val == "Benigno")).sum())

print(f"\nRecall (não-Benigno / proteção contra FN): {recall_score(y_true_val, nao_benigno_pred):.4f}")
print(f"Falsos Negativos (Malignos classificados como Benigno): {fn_benigno}")
print(f"Precision (Maligno direto >= t_alto): {precision_score(y_true_val, maligno_direto_pred, zero_division=0):.4f}")

# Tabela cruzada: Classe Real x Faixa Prevista
tabela = pd.crosstab(
    pd.Series(np.where(y_true_val == 1, "Maligno (Real)", "Benigno (Real)"), name="Real"),
    pd.Series(faixas_val, name="Previsto"),
    margins=True
)
display(tabela)

In [ ]:
import inspect

print("=== Hiperparâmetros do modelo ===")
print(model.hparams)

print("\n=== Código do SuperpixelGraphExplainer ===")
print(inspect.getsource(explainer.__class__))

In [ ]:
nome, caminho = amostras_visuais[-1]  # ex: Maligno 2
img = Image.open(caminho).convert("RGB")
prob_explainer = explainer.explain(img, seed=42)["probabilidade"]

W, H = explainer.input_width, explainer.input_height
device = next(model.parameters()).device
model.eval()

tfm_sem_norm = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
])

tfm_com_norm = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

with torch.no_grad():
    for rotulo, tfm in [("Sem Normalize", tfm_sem_norm), ("Com Normalize", tfm_com_norm)]:
        x = tfm(img).unsqueeze(0).to(device)
        out = model(x)
        prob_raw = out.view(-1).item()
        prob_sig = torch.sigmoid(out.view(-1)).item()
        print(f"{rotulo:15s} -> raw: {prob_raw:.4f} | com sigmoid: {prob_sig:.4f}")

print(f"Explainer       -> prob: {prob_explainer:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

def definir_cortes_clinicos(y_true, y_prob, recall_alvo=0.95, precision_alvo=0.85, plotar=True):
    prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
    prec_t, rec_t = prec[:-1], rec[:-1]

    # 1. t_baixo: maior limiar que ainda garante recall >= recall_alvo (evita FN)
    idx_rec = np.where(rec_t >= recall_alvo)[0]
    t_baixo = float(thresholds[idx_rec[-1]]) if len(idx_rec) > 0 else float(thresholds[np.argmax(rec_t)])

    # 2. t_alto: menor limiar (> t_baixo) que atinge precision >= precision_alvo
    idx_prec = np.where((thresholds > t_baixo) & (prec_t >= precision_alvo))[0]
    if len(idx_prec) > 0:
        t_alto = float(thresholds[idx_prec[0]])
    else:
        t_alto = float(thresholds[-1])

    faixas = np.where(
        y_prob >= t_alto, "Maligno",
        np.where(y_prob >= t_baixo, "Inconclusivo", "Benigno")
    )

    if plotar:
        plt.figure(figsize=(9, 5))
        plt.plot(thresholds, prec_t, label="Precision (classe Maligno)", linewidth=2)
        plt.plot(thresholds, rec_t, label="Recall (classe Maligno)", linewidth=2)
        plt.axvline(t_baixo, color="green", linestyle=":", linewidth=2, label=f"t_baixo = {t_baixo:.3f} (Rec >= {recall_alvo})")
        plt.axvline(t_alto, color="red", linestyle=":", linewidth=2, label=f"t_alto = {t_alto:.3f} (Prec >= {precision_alvo})")
        plt.axvspan(0.0, t_baixo, color="green", alpha=0.06)
        plt.axvspan(t_baixo, t_alto, color="orange", alpha=0.08)
        plt.axvspan(t_alto, 1.0, color="red", alpha=0.06)
        plt.title(f"Cortes Clínicos — Recall alvo: {recall_alvo:.0%} | Precision alvo: {precision_alvo:.0%}")
        plt.xlabel("Limiar de probabilidade")
        plt.ylabel("Pontuação")
        plt.xlim(0.0, 1.0)
        plt.ylim(0.0, 1.05)
        plt.grid(True, alpha=0.3)
        plt.legend(loc="best")
        plt.tight_layout()
        plt.show()

    return t_baixo, t_alto, faixas

# Compara cenários de Precision mínima para t_alto
resumo = []
for p_alvo in [0.75, 0.80, 0.85, 0.90]:
    tb, ta, fx = definir_cortes_clinicos(y_true_val, y_prob_val, recall_alvo=0.95, precision_alvo=p_alvo, plotar=False)
    resumo.append({
        "precision_alvo": p_alvo,
        "t_baixo": round(tb, 3),
        "t_alto": round(ta, 3),
        "FN_em_Benigno": int(((y_true_val == 1) & (fx == "Benigno")).sum()),
        "FP_em_Maligno": int(((y_true_val == 0) & (fx == "Maligno")).sum()),
        "qtd_Inconclusivo": int((fx == "Inconclusivo").sum()),
        "pct_Inconclusivo": f"{100 * (fx == 'Inconclusivo').mean():.1f}%",
        "precision_Maligno": round(precision_score(y_true_val, (y_prob_val >= ta).astype(int), zero_division=0), 4),
        "recall_nao_Benigno": round(recall_score(y_true_val, (y_prob_val >= tb).astype(int), zero_division=0), 4),
    })

display(pd.DataFrame(resumo))

# Plota e mostra a tabela cruzada detalhada para precision_alvo = 0.85
t_baixo, t_alto, faixas_val = definir_cortes_clinicos(y_true_val, y_prob_val, recall_alvo=0.95, precision_alvo=0.85, plotar=True)

tabela = pd.crosstab(
    pd.Series(np.where(y_true_val == 1, "Maligno (Real)", "Benigno (Real)"), name="Real"),
    pd.Series(faixas_val, name="Previsto"),
    margins=True
)
display(tabela)

In [ ]:
t_baixo, t_alto

In [ ]:
import importlib
import src.explainability
importlib.reload(src.explainability)

explainer = src.explainability.SuperpixelGraphExplainer(
    model=model,
    t_baixo=t_baixo,
    t_alto=t_alto,
    device=str(device),
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

linhas = []
fig, axes = plt.subplots(len(amostras_visuais), 2, figsize=(10, 4 * len(amostras_visuais)))

for i, (nome, caminho) in enumerate(amostras_visuais):
    img_orig = Image.open(caminho).convert("RGB")
    resultado = explainer.explain(img_orig, seed=42)

    prob = resultado["probabilidade"]
    classificacao = resultado["classificacao"]
    n_nos = resultado["num_superpixels"]
    n_arestas = resultado["num_arestas"]
    drop_top5 = resultado["fidelity_drop_top5"]
    overlay = resultado["overlay_image"]

    # Monta os dados da tabela final
    linhas.append({
        "Amostra": nome,
        "Probabilidade": round(prob, 4),
        "Classificação": classificacao,
        "Nós": n_nos,
        "Arestas": n_arestas,
        "Drop Top-5": round(drop_top5, 4),
    })

    # Identifica a origem da imagem automaticamente pelo caminho
    origem_img = "augmented_data" if "augmented" in str(caminho) else "original_data"

    # Coluna da esquerda: Imagem de entrada
    axes[i, 0].imshow(img_orig)
    axes[i, 0].set_title(f"{nome} (Img: {origem_img})", fontsize=11)
    axes[i, 0].axis("off")

    # Coluna da direita: Overlay com a classificação calibrada
    axes[i, 1].imshow(overlay)
    axes[i, 1].set_title(
        f"Pred: {classificacao} | Prob: {prob:.2f} | Nós: {n_nos} | Drop Top-5: {drop_top5:.2f}",
        fontsize=10
    )
    axes[i, 1].axis("off")

# 1. Tabela final
df_resultados_finais = pd.DataFrame(linhas)
display(df_resultados_finais)

# 2. Painel visual atualizado
fig.suptitle(
    f"Explicabilidade via Superpixels (SLIC + Ridge) — Limiares: t_baixo={t_baixo:.3f} | t_alto={t_alto:.3f}",
    fontsize=12,
    y=1.01
)
plt.tight_layout()
plt.savefig("resultados_superpixels_overlay.png", dpi=200, bbox_inches="tight")
plt.show()

## Analse do dataset que possui as mascaras

In [ ]:
from pathlib import Path
from collections import Counter

PASTA_DRYAD = Path("../data/dataset_mascaras")  # ajuste para a pasta onde você extraiu

todos_arquivos = [p for p in PASTA_DRYAD.rglob("*") if p.is_file()]
print(f"Total de arquivos: {len(todos_arquivos)}")
print("Extensões encontradas:", Counter(p.suffix.lower() for p in todos_arquivos))

print("\nSubpastas:")
for d in sorted({p.parent for p in todos_arquivos}):
    print(" -", d.relative_to(PASTA_DRYAD))

print("\nPrimeiros 10 arquivos:")
for p in sorted(todos_arquivos)[:10]:
    print(" -", p.relative_to(PASTA_DRYAD))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.segmentation import find_boundaries

# 1. Ajuste para os caminhos do dataset novo
PASTA_IMAGENS = Path("../data/dataset_mascaras/images")
PASTA_MASCARAS = Path("../data/dataset_mascaras/masks")
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# Pareia imagem <-> máscara pelo nome base (stem)
mapa_mascaras = {p.stem: p for p in PASTA_MASCARAS.rglob("*") if p.suffix.lower() in EXTS}
pares = [
    (img_p, mapa_mascaras[img_p.stem])
    for img_p in sorted(PASTA_IMAGENS.rglob("*"))
    if img_p.suffix.lower() in EXTS and img_p.stem in mapa_mascaras
]
print(f"Pares (imagem, máscara) encontrados: {len(pares)}")

W, H = explainer.input_width, explainer.input_height
registros = []
exemplos_plot = []

for img_path, mask_path in pares:
    img = Image.open(img_path).convert("RGB")
    # Usa NEAREST na máscara para manter os valores binários sem borrar bordas
    mask_img = Image.open(mask_path).convert("L").resize((W, H), Image.NEAREST)
    mask_bin = np.asarray(mask_img) > 127

    if not np.any(mask_bin):
        continue

    res = explainer.explain(img, seed=42)
    hmap = res["heatmap"]
    energia_total = float(hmap.sum())

    if energia_total > 0:
        eim = float(hmap[mask_bin].sum() / energia_total)
        pico_y, pico_x = np.unravel_index(np.argmax(hmap), hmap.shape)
        pointing_hit = int(mask_bin[pico_y, pico_x])
    else:
        eim, pointing_hit = 0.0, 0

    area_lesao = float(mask_bin.mean())
    razao_foco = eim / area_lesao if area_lesao > 0 else 0.0

    registros.append({
        "arquivo": img_path.name,
        "probabilidade": round(res["probabilidade"], 4),
        "classificacao": res["classificacao"],
        "drop_top5": round(res["fidelity_drop_top5"], 4),
        "area_lesao": round(area_lesao, 4),
        "EIM": round(eim, 4),
        "razao_foco": round(razao_foco, 2),
        "pointing_hit": pointing_hit,
    })

    # Guarda até 4 exemplos para visualização
    if len(exemplos_plot) < 4:
        exemplos_plot.append((img_path.name, img.resize((W, H)), mask_bin, res))

df_mascaras = pd.DataFrame(registros)
display(df_mascaras.head(10))

print("\n=== Resumo Quantitativo (XAI vs. Máscaras Reais) ===")
print(f"EIM médio: {df_mascaras['EIM'].mean():.4f} ± {df_mascaras['EIM'].std():.4f}")
print(f"Razão de Foco média (> 1.0 = acima do acaso): {df_mascaras['razao_foco'].mean():.2f}x")
print(f"Pointing Game (Acerto do pico na lesão): {100 * df_mascaras['pointing_hit'].mean():.1f}%")
print(f"Drop Top-5 médio: {df_mascaras['drop_top5'].mean():.4f}")

# Painel visual: Imagem + contorno da máscara real | Overlay XAI + contorno da máscara real
fig, axes = plt.subplots(len(exemplos_plot), 2, figsize=(10, 4 * len(exemplos_plot)))
if len(exemplos_plot) == 1:
    axes = np.array([axes])

for i, (nome, img_res, mask_bin, res) in enumerate(exemplos_plot):
    contorno = find_boundaries(mask_bin, mode="thick")

    img_cont = np.asarray(img_res).copy()
    img_cont[contorno] = [0, 255, 0]  # contorno verde da ferida real

    overlay_cont = res["overlay_image"].copy()
    overlay_cont[contorno] = [0, 255, 0]

    axes[i, 0].imshow(img_cont)
    axes[i, 0].set_title(f"{nome} — Máscara Real (verde)", fontsize=10)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(overlay_cont)
    axes[i, 1].set_title(
        f"Pred: {res['classificacao']} ({res['probabilidade']:.2f}) | EIM: {df_mascaras.loc[i, 'EIM']:.2f}",
        fontsize=10
    )
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("validacao_xai_mascaras.png", dpi=200, bbox_inches="tight")
plt.show()